# PaperMono · audio lab

Explore the original spectrum and spectrogram, then listen to a **causal biquad filter** on the same audio.
No firmware changes; no files written unless you explicitly enable the export cell.

Run top to bottom. Start with a short preview containing speech and room noise.


In [ ]:
from pathlib import Path
import sys
import json
import wave

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display, Markdown
import ipywidgets as widgets

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/audio/analyze_recording.py").exists())
sys.path.insert(0, str(ROOT / "scripts/audio"))
import importlib
import analyze_recording
importlib.reload(analyze_recording)  # Pick up loader changes in an already-running kernel.
from analyze_recording import load_audio, spectrum, spectrogram
%matplotlib inline
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": False})


## 1. Choose a recording

A session directory joins numbered WAVs in order, omitting intentional pauses. A merged WAV works directly.
Change this path to your recording; it is never modified.

**PCM16 and PaperMono IMA ADPCM WAVs** are supported, including 16, 32 and 48 kHz recordings.
The sample rate comes from the WAV header; plots, playback and filter design use that rate automatically.
No resampling is performed. PCM loading does not need FFmpeg; ADPCM does (`brew install ffmpeg` on macOS).
Declared ADPCM padding is removed. You can load a session directory directly, without merging first.

After changing the recording, rerun from this cell downward, including the filter controls.
A 32/48 kHz source produces coefficients for that rate: do not paste them into 16 kHz firmware.


In [ ]:
RECORDING = Path.home() / "Downloads/session-000001-merged.wav"
if not RECORDING.exists():
    raise FileNotFoundError(f"Set RECORDING to your WAV or session directory: {RECORDING}")
latest = None  # Invalidate the previous recording's filter/export result.
original, sample_rate, boundaries, file_count = load_audio(RECORDING)
duration = len(original) / sample_rate
print(f"{file_count} file(s), {duration:.3f} seconds, {sample_rate} Hz; Nyquist {sample_rate/2:g} Hz")
print(f"Peak: {20*np.log10(max(np.max(np.abs(original)), 1e-8)):.1f} dBFS")


In [ ]:
MIN_PLOT_HZ = 20.0  # DC (0 Hz) cannot be displayed on a logarithmic axis.

def plot_fft(ax, samples, label):
    frequencies, db = spectrum(samples, sample_rate)
    frequencies, db = frequencies[1:], db[1:]  # Exclude DC.
    # Peak-preserving display reduction; FFT still uses the entire supplied signal.
    step = max(1, int(np.ceil(len(db) / 10000)))
    starts = np.arange(0, len(db), step)
    ax.plot(frequencies[starts], np.maximum.reduceat(db, starts), lw=0.7, label=label)
    ax.set(xlabel="Frequency (Hz)", ylabel="Amplitude (dBFS)", xscale="log", xlim=(MIN_PLOT_HZ, sample_rate/2), ylim=(-140, 0))
    ax.grid(alpha=0.2)
    ax.legend()


def plot_heatmap(ax, samples, title, start=0):
    freq, times, db, *_ = spectrogram(samples, sample_rate, 64, 16)
    edges = np.r_[0, (freq[:-1] + freq[1:])/2, sample_rate/2]
    mesh = ax.pcolormesh(times + start, edges[1:], db[1:], shading="flat", cmap="magma", vmin=-110, vmax=-30,
                         rasterized=True)
    ax.set(xlabel="Recorded time (seconds)", ylabel="Frequency (Hz)", title=title,
           xlim=(start, start + len(samples)/sample_rate),
           yscale="log", ylim=(MIN_PLOT_HZ, sample_rate/2))
    return mesh

fig, axes = plt.subplots(2, 1, figsize=(12, 8), layout="constrained")
plot_fft(axes[0], original, "Whole-recording FFT · Hann window · DC removed")
mesh = plot_heatmap(axes[1], original, "Original · 64 ms windows, 16 ms hop")
for boundary in boundaries:
    axes[1].axvline(boundary, color="cyan", lw=0.5)
fig.colorbar(mesh, ax=axes[1], label="dBFS/Hz")
plt.show()


## 2. Filter and compare

Choose **Butterworth, Chebyshev I, Chebyshev II or Elliptic**, and order **2, 4 or 6 per enabled edge**.
Each family works with high-pass only, low-pass only, both, or neither (bypass). Both edges together double
that order: order 4 per edge means 8 total, implemented as four biquads. Fourth order preserves your current setting.

Ripple/attenuation defaults above the design function are 0.5 dB / 40 dB. They apply where the family uses them.
**Edge Hz is not identical across families:** Butterworth uses −3 dB; Chebyshev I and elliptic use the passband
ripple boundary; Chebyshev II uses the stopband attenuation boundary. Inspect the response curve before treating
identical slider values as equivalent bandwidths. These are cascaded HP/LP designs, not a separate bandpass mode.

Designed poles must lie inside the unit circle, and non-finite coefficients/output are rejected. Stable filters
can still ring, overshoot or remove useful speech. Original samples remain untouched; this is not a guarantee of
pleasant sound or embedded float32 stability. Start playback at a comfortable volume.

The full recording is filtered forward once, keeping state throughout. Listening/plots use your preview range.
Gain is applied **after** filtering. We report overload without clipping the analysis signal; playback uses a
shared attenuation for both versions if necessary. There is no independent loudness normalization.
Click **Apply & compare** after editing the controls.


In [ ]:
FILTER_FAMILIES = {"Butterworth": "butter", "Chebyshev I": "cheby1",
                   "Chebyshev II": "cheby2", "Elliptic": "ellip"}
PASSBAND_RIPPLE_DB = 0.5
STOPBAND_ATTENUATION_DB = 40.0


def design_biquads(rate, highpass, lowpass, high_hz, low_hz,
                   family="butter", edge_order=4):
    if family not in FILTER_FAMILIES.values() or edge_order not in (2, 4, 6):
        raise ValueError("Choose a listed filter family and order")
    if not np.isfinite(rate) or rate <= 0:
        raise ValueError("Sample rate must be positive and finite")
    if highpass and not 0 < high_hz < rate/2:
        raise ValueError("High-pass cutoff must be below Nyquist")
    if lowpass and not 0 < low_hz < rate/2:
        raise ValueError("Low-pass cutoff must be below Nyquist")
    if highpass and lowpass and high_hz >= low_hz:
        raise ValueError("High-pass cutoff must be lower than low-pass cutoff")
    sections = []
    for enabled, cutoff, kind in [(highpass, high_hz, "highpass"), (lowpass, low_hz, "lowpass")]:
        if enabled:
            sections.append(signal.iirfilter(edge_order, cutoff, btype=kind, ftype=family,
                            rp=PASSBAND_RIPPLE_DB, rs=STOPBAND_ATTENUATION_DB, fs=rate, output="sos"))
    sos = np.vstack(sections) if sections else np.empty((0, 6))
    if not np.isfinite(sos).all():
        raise ValueError("Non-finite filter coefficients; choose less extreme settings")
    if len(sos) and np.any(np.abs(signal.sos2zpk(sos)[1]) >= 1):
        raise ValueError("Unstable filter poles; choose less extreme settings")
    return sos


latest = None

def compare(highpass=True, lowpass=False, high_hz=100.0, low_hz=6000.0, gain_db=0.0,
            start_seconds=0.0, preview_seconds=20.0, family="butter", edge_order=4):
    global latest
    latest = None  # An invalid new selection must not export a stale result.
    try:
        sos = design_biquads(sample_rate, highpass, lowpass, high_hz, low_hz, family, edge_order)
        if not np.isfinite(gain_db):
            raise ValueError("Gain must be finite")
        first = max(0, round(start_seconds * sample_rate))
        last = min(len(original), first + round(preview_seconds * sample_rate))
        if last - first < 4:
            raise ValueError("Choose a preview within the recording")
    except ValueError as error:
        print(error)
        return
    filtered = signal.sosfilt(sos, original) if len(sos) else original.copy()
    filtered *= 10 ** (gain_db / 20)
    if not np.isfinite(filtered).all():
        print("Non-finite output rejected; reduce gain or choose less extreme filter settings.")
        return
    a, b = original[first:last], filtered[first:last]
    playback_scale = max(1.0, np.max(np.abs(a)), np.max(np.abs(b)))
    print(f"{len(sos)} biquad(s) · {len(sos)*2} total order · "
          f"{np.count_nonzero(np.abs(filtered) > 1)} samples exceed full scale after gain")
    print(f"Both players attenuated by {20*np.log10(playback_scale):.2f} dB; no independent normalization.")
    display(Markdown("**Original**"), Audio(a/playback_scale, rate=sample_rate, normalize=False))
    display(Markdown("**Filtered**"), Audio(b/playback_scale, rate=sample_rate, normalize=False))
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), layout="constrained")
    f = np.geomspace(MIN_PLOT_HZ, sample_rate/2, 2048)
    response = signal.sosfreqz(sos, worN=f, fs=sample_rate)[1] if len(sos) else np.ones_like(f)
    axes[0].plot(f, 20*np.log10(np.maximum(np.abs(response), 1e-8)) + gain_db)
    axes[0].set(title="Filter response + gain", xlabel="Frequency (Hz)", ylabel="Gain (dB)", ylim=(-80, 18),
                xscale="log", xlim=(MIN_PLOT_HZ, sample_rate/2))
    axes[0].grid(alpha=0.2)
    plot_fft(axes[1], a, "Original preview")
    plot_fft(axes[1], b, "Filtered preview")
    plt.show()
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), layout="constrained", sharex=True, sharey=True)
    plot_heatmap(axes[0], a, "Original preview", first/sample_rate)
    mesh = plot_heatmap(axes[1], b, "Filtered preview · same color scale", first/sample_rate)
    fig.colorbar(mesh, ax=list(axes), label="dBFS/Hz")
    plt.show()
    latest = {"audio": filtered, "sos": sos, "sample_rate": sample_rate, "gain_db": gain_db,
              "highpass": highpass, "lowpass": lowpass, "high_hz": high_hz, "low_hz": low_hz,
              "family": family, "edge_order": edge_order, "passband_ripple_db": PASSBAND_RIPPLE_DB,
              "stopband_attenuation_db": STOPBAND_ATTENUATION_DB, "source": str(RECORDING)}
    print("SOS coefficients: [b0, b1, b2, a0, a1, a2] (SciPy convention)")
    print(sos)

controls = widgets.interact_manual.options(manual_name="Apply & compare")(
    compare,
    family=widgets.Dropdown(options=FILTER_FAMILIES, value="ellip", description="Family"),
    edge_order=widgets.Dropdown(options=[2, 4, 6], value=4, description="Edge order"),
    highpass=widgets.Checkbox(value=True, description="High-pass"),
    lowpass=widgets.Checkbox(value=True, description="Low-pass"),
    high_hz=widgets.FloatSlider(value=150, min=20, max=min(1000, sample_rate/2-1), step=10, description="HP Hz", continuous_update=False),
    low_hz=widgets.FloatSlider(value=min(4500, sample_rate*0.45), min=1000, max=sample_rate/2-1, step=50, description="LP Hz", continuous_update=False),
    gain_db=widgets.FloatSlider(value=0, min=-12, max=12, step=0.5, description="Gain dB", continuous_update=False),
    start_seconds=widgets.FloatSlider(value=0, min=0, max=max(0, duration-0.1), step=0.1, description="Start sec", continuous_update=False),
    preview_seconds=widgets.FloatSlider(value=min(20, duration), min=min(0.1, duration), max=min(120, duration), step=0.1, description="Length sec", continuous_update=False),
)


## 3. Optional export

Enable only when you want an artifact. Exports the **full** filtered recording and settings under `outputs/`.
Overloaded output is rejected rather than silently clipped: reduce gain and compare again. Existing files are
not overwritten. The filter settings are for this sample rate and use SciPy’s coefficient/sign convention.
They are not automatically installed into the firmware.


In [ ]:
EXPORT = False
EXPORT_NAME = "filtered-trial-08"  # Change for each trial.
if EXPORT:
    assert latest is not None, "Click Apply & compare first"
    assert Path(EXPORT_NAME).name == EXPORT_NAME and EXPORT_NAME not in (".", ".."), "Use a simple filename"
    samples = latest["audio"]
    print("All finite:", np.isfinite(samples).all())
    print("Minimum:", np.min(samples))
    print("Maximum:", np.max(samples))
    print("Absolute peak:", np.max(np.abs(samples)))
    print("Limit:", 32767 / 32768)
    valid = bool(
        np.isfinite(samples).all()
        and np.max(np.abs(samples)) <= 32767 / 32768
    )
    print("Passes export check:", valid)
    assert valid, "Reduce gain before exporting"
    destination = ROOT / "notebooks/audio/outputs"
    destination.mkdir(exist_ok=True)
    wav_path = destination / f"{EXPORT_NAME}.wav"
    settings_path = destination / f"{EXPORT_NAME}.json"
    assert not wav_path.exists() and not settings_path.exists(), "Choose a new trial name"
    with wav_path.open("xb") as file, wave.open(file, "wb") as wav:
        wav.setparams((1, 2, latest["sample_rate"], 0, "NONE", "not compressed"))
        wav.writeframes(np.rint(samples*32768).astype("<i2").tobytes())
    settings = {key: value for key, value in latest.items() if key != "audio"}
    settings["sos"] = settings["sos"].tolist()
    settings_path.write_text(json.dumps(settings, indent=2) + "\n")
    print(wav_path)


## Small numerical check

Known tones verify passband/stopband behavior, and chunked filtering must match a single continuous pass.
This checks the desktop experiment, not on-device performance.


In [ ]:
def check_filters():
    rate = sample_rate
    rng = np.random.default_rng(42)
    x = rng.normal(0, 0.1, 16000)
    for family in FILTER_FAMILIES.values():
        for order in (2, 4, 6):
            for hp, lp in ((False, False), (True, False), (False, True), (True, True)):
                sos = design_biquads(rate, hp, lp, 100, 6000, family, order)
                assert len(sos) == (hp + lp) * order // 2
                if not len(sos):
                    continue
                assert np.max(np.abs(signal.sos2zpk(sos)[1])) < 1
                y = signal.sosfilt(sos, x)
                assert np.isfinite(y).all()
                state = np.zeros((len(sos), 2))
                chunks = []
                for start in range(0, len(x), 1024):
                    chunk, state = signal.sosfilt(sos, x[start:start+1024], zi=state)
                    chunks.append(chunk)
                np.testing.assert_allclose(np.concatenate(chunks), y, atol=1e-12)
    sos = design_biquads(rate, True, True, 100, 6000)
    _, h = signal.sosfreqz(sos, worN=np.array([10, 1000, rate * 0.49375]), fs=rate)
    assert abs(h[0]) < 0.02 and abs(h[1]) > 0.95 and abs(h[2]) < 0.02
    for cutoff in (0, rate/2, np.nan):
        try:
            design_biquads(rate, True, False, cutoff, 6000)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid cutoff accepted")
    fig, axes = plt.subplots(2, 1)
    plot_fft(axes[0], x, "Scale check")
    plot_heatmap(axes[1], x, "Scale check")
    assert axes[0].get_xscale() == axes[1].get_yscale() == "log"
    assert axes[0].get_xlim()[0] > 0 and axes[1].get_ylim()[0] > 0
    plt.close(fig)
    print("Passed: all families/orders/modes, stable poles, finite output, continuous state and invalid cutoffs")

check_filters()
